# Delivery Demand Forecasting

## 7-day hourly forecasts across a delivery network

This portfolio notebook summarizes an end-to-end forecasting case study using the public **Delivery Center: Food & Goods Orders in Brazil** dataset.

**Target:** hourly delivery requests by Hub × Segment.

**Scale:** 368,999 orders, 951 stores, 32 hubs, and 58 observed Hub × Segment series.

The workflow compares a weekly seasonal baseline, local Prophet models, and a global XGBoost model under leakage-safe rolling-origin validation. It then connects the forecasts to an operational capacity-planning prototype.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw" / "delivery_center"

orders = pd.read_csv(RAW_DIR / "orders.csv")
stores = pd.read_csv(RAW_DIR / "stores.csv")
hubs = pd.read_csv(RAW_DIR / "hubs.csv")
deliveries = pd.read_csv(RAW_DIR / "deliveries.csv")
drivers = pd.read_csv(RAW_DIR / "drivers.csv")

## Forecasting target and validation

An order creation is treated as an incoming delivery request. Order-level data is aggregated to a complete hourly grid for every observed Hub × Segment pair.

The business horizon is **168 hours (7 days)**. Evaluation uses rolling forecast origins so each model is trained only on information available before the forecast window.

A particularly important modeling choice is feature availability: short lags such as `lag_1` and `lag_24` are not valid inputs for every timestamp of a direct 168-hour forecast. The global model therefore uses weekly-or-longer lags that are known throughout the full horizon.

In [ ]:
def metrics(y, pred):
    y = np.asarray(y)
    pred = np.asarray(pred)
    return pd.Series({
        "MAE": np.mean(np.abs(y - pred)),
        "wMAPE": np.abs(y - pred).sum() / np.abs(y).sum(),
        "Bias": np.mean(pred - y),
    })

# Strong operational benchmark:
# y_hat(t) = y(t - 168 hours)
hourly["seasonal_naive"] = hourly.groupby(
    ["hub_id", "store_segment"]
)["requests"].shift(168)

# Leakage-safe global features use lags available across the entire horizon.
for lag in [168, 336, 504, 672]:
    hourly[f"lag_{lag}"] = hourly.groupby(
        ["hub_id", "store_segment"]
    )["requests"].shift(lag)

## Model comparison

Three approaches were evaluated on common rolling-origin windows:

1. **Seasonal Naive:** previous week's demand at the same hour.
2. **Local Prophet:** an independent time-series model for each Hub × Segment.
3. **Global XGBoost:** a single model sharing information across series, with categorical location/segment features, calendar variables, and leakage-safe historical lags.

In [ ]:
results = pd.DataFrame({
    "Model": ["Global XGBoost", "Seasonal Naive", "Local Prophet"],
    "MAE": [1.4088, 1.5481, 1.9521],
    "wMAPE": [0.5186, 0.5699, 0.7186],
    "Bias": [0.2304, 0.0928, 0.4220],
})

results.style.format({
    "MAE": "{:.4f}",
    "wMAPE": "{:.2%}",
    "Bias": "{:.4f}",
})

## Findings

**Global XGBoost achieved the strongest overall point accuracy:** MAE 1.4088 and wMAPE 51.86%. This reduced weighted absolute error by roughly 9% versus seasonal naive and 28% versus local Prophet.

The result is more nuanced by segment and volume:

- **FOOD:** XGBoost 37.3% wMAPE vs. 43.0% seasonal naive and 55.2% Prophet.
- **Sparse GOOD series:** all models are substantially weaker and percentage metrics become unstable near zero.
- **Low-volume series:** the simple weekly baseline can outperform the global ML model.
- **High-volume series:** XGBoost provides the clearest improvement.

The seasonal baseline also has lower aggregate bias than XGBoost. For an operational system, that matters because systematic overforecasting can translate directly into excess capacity cost.

## From forecasting to capacity planning

The case study extends the forecast into a simple driver-capacity proxy. Historical out-of-sample forecast residuals represent demand uncertainty and observed deliveries provide a productivity estimate.

For each hub-hour, candidate driver counts are evaluated against expected driver cost plus a shortage penalty. The resulting sensitivity analysis makes the service/cost trade-off explicit: increasing the assumed shortage cost leads to more allocated capacity and higher fulfillment, but also greater excess capacity.

This is an operational prototype rather than a production workforce optimizer. A production version would incorporate scheduled availability, labor rules, idle capacity, geographic repositioning, and calibrated SLA/cancellation economics.

## Takeaways

This project demonstrates:

- relational data preparation and hourly demand construction;
- multi-series forecasting across a heterogeneous network;
- leakage-aware feature engineering for multi-step horizons;
- strong-baseline discipline;
- rolling-origin validation rather than reliance on a single holdout;
- local statistical forecasting versus global machine learning;
- diagnostics by segment and volume;
- translation of forecast uncertainty into an operational decision framework.

See **02_backtesting_and_window_sensitivity.ipynb** for the temporal-stability and training-history analysis.